# Photon transfer curve (PTC) diagnostic plot

Load a [`PhotonTransferCurveDataset`](https://pipelines.lsst.io/py-api/lsst.ip.isr.PhotonTransferCurveDataset.html) and plot measured \(C_{00}\) vs mean signal \(\mu\) (ADU), with:

- **Preliminary fit (Astier+19 Eq. 16)** — exponential approximation only (`funcAstier`)
- **Preliminary fit + saturation roll-off** — Eq. 16 plus the piecewise exponential roll-off term (`funcAstierWithRolloff`, same as `lsst.cp.pipe`)
- **Final fit (Astier+19 Eq. 20)** — full covariance model variance \(C_{00}\) from `evalPtcModel` when `ptcFitType` is `FULLCOVARIANCE`

Set `PTC_PATH` below (or export it in the shell) to your `*_ptc.fits` calibration file.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from lsst.ip.isr import PhotonTransferCurveDataset

# Path to a PhotonTransferCurveDataset FITS product (override as needed).
PTC_PATH = os.environ.get(
    "PTC_PATH",
    "/path/to/your/detector_ptc.fits",
)
AMP = "C15"  # amplifier name in the dataset
XLIM = (0, 100_000)
YLIM = (0, 40_000)

In [ ]:
AMP = ""  # amplifier name in the dataset
XLIM = (0, 100_000)
YLIM = (0, 40_000)

# Match Rubin DM / cp_pipe when available; local fallbacks otherwise.
try:
    from lsst.cp.pipe.utils import funcAstier, funcAstierWithRolloff
except ImportError:
    def funcAstier(pars, x):
        a00, gain, noise_squared = pars[:3]
        return (
            0.5 / (a00 * gain * gain) * (np.exp(2 * a00 * x * gain) - 1)
            + noise_squared / (gain * gain)
        )

    def _ptc_rolloff_model(params, mu):
        mu_turn, tau = params
        return np.where(mu < mu_turn, 0.0, np.exp(-(mu - mu_turn) / tau) - 1.0)

    def funcAstierWithRolloff(pars, x):
        return funcAstier(pars[:-2], x) - _ptc_rolloff_model(pars[-2:], x)

def _prelim_pars(ptc, amp):
    """Three-parameter EXPAPPROXIMATION fit stored after the pre-fit stage."""
    pars = np.asarray(ptc.ptcFitPars[amp], dtype=float)
    if pars.size != 3 or not np.all(np.isfinite(pars)):
        raise ValueError(
            f"{amp}: expected finite ptcFitPars of length 3 (a00, gain, n00); got {pars!r}"
        )
    return pars


def _final_c00_model(ptc, amp, mu):
    """Variance C_00 from the final Astier+19 Eq. 20 fit (ADU^2)."""
    mu = np.asarray(mu, dtype=float)
    if ptc.ptcFitType in ("FULLCOVARIANCE", "FULLCOVARIANCE_NO_B"):
        cov = ptc.evalPtcModel(mu)[amp]
        return np.asarray(cov)[:, 0, 0]
    if ptc.ptcFitType == "EXPAPPROXIMATION":
        return funcAstier(_prelim_pars(ptc, amp), mu)
    raise RuntimeError(f"Unsupported ptcFitType for model overlay: {ptc.ptcFitType!r}")


def plot_ptc_amp(
    ptc,
    amp,
    *,
    xlim=XLIM,
    ylim=YLIM,
    figsize=(6.5, 4.2),
):
    pars = _prelim_pars(ptc, amp)
    mu_roll = ptc.ptcRolloff[amp]
    tau = ptc.ptcRolloffTau[amp]
    has_roll = np.isfinite(mu_roll) and np.isfinite(tau)

    raw_m = np.asarray(ptc.rawMeans[amp], dtype=float)
    raw_v = np.asarray(ptc.rawVars[amp], dtype=float)
    fin_m = np.asarray(ptc.finalMeans[amp], dtype=float)
    fin_v = np.asarray(ptc.finalVars[amp], dtype=float)
    used = np.isfinite(fin_m) & np.isfinite(fin_v)

    mu_hi = min(xlim[1], np.nanmax(raw_m) * 1.02)
    mu_fine = np.linspace(0.0, mu_hi, 800)

    y_exp = funcAstier(pars, mu_fine)
    y_roll = funcAstierWithRolloff(np.r_[pars, mu_roll, tau], mu_fine) if has_roll else None
    y_final = _final_c00_model(ptc, amp, mu_fine)

    try:
        plt.style.use("seaborn-v0_8-whitegrid")
    except OSError:
        pass

    fig, ax = plt.subplots(figsize=figsize, constrained_layout=True)

    ax.scatter(raw_m, raw_v, s=4, alpha=0.35, c="0.55", label="All pairs", rasterized=True)
    ax.scatter(
        fin_m[used],
        fin_v[used],
        s=12,
        alpha=0.8,
        c="0.12",
        edgecolors="none",
        label="Used in final fit",
        rasterized=True,
    )

    ax.plot(mu_fine, y_exp, color="C0", lw=2.0, label="Preliminary: Eq. 16 (no roll-off)")
    if has_roll:
        ax.plot(
            mu_fine,
            y_roll,
            color="C3",
            lw=2.0,
            label="Preliminary: Eq. 16 + saturation roll-off",
        )
    ax.plot(
        mu_fine,
        y_final,
        color="C2",
        lw=2.0,
        ls="--",
        label=f"Final: {ptc.ptcFitType} (Eq. 20 $C_{{00}}$)",
    )

    if np.isfinite(ptc.ptcTurnoff[amp]):
        ax.axvline(
            ptc.ptcTurnoff[amp],
            ls="--",
            color="0.35",
            lw=1.2,
            alpha=0.85,
            label="PTC turnoff",
        )
    if has_roll:
        ax.axvline(
            mu_roll,
            ls=":",
            color="0.35",
            lw=1.5,
            alpha=0.85,
            label="PTC roll-off (stored)",
        )

    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_xlabel(r"Mean signal $\mu$ (ADU)")
    ax.set_ylabel(r"Variance $C_{00}$ (ADU$^2$)")
    ax.set_title(f"Photon transfer curve — {amp}")
    ax.legend(loc="upper left", fontsize="small", frameon=True)
    ax.xaxis.set_major_formatter(
        plt.FuncFormatter(lambda v, _: f"{v/1000:.0f}k" if v >= 1000 else f"{v:.0f}")
    )
    return fig, ax

fig, ax = plot_ptc_amp(ptc, AMP)
# fig.savefig(Path("figures") / f"ptc_{AMP}.pdf", dpi=200)
plt.show()

In [ ]:
fig, ax = plot_ptc_amp(ptc, AMP)
# fig.savefig(Path("figures") / f"ptc_{AMP}.pdf", dpi=200)
plt.show()

### Minimal version (your original snippet + model lines)

Equivalent to the cells above, without the styling helper:

In [ ]:
amp = AMP
pars = np.asarray(ptc.ptcFitPars[amp], dtype=float)
tau = ptc.ptcRolloffTau[amp]
mu_star = ptc.ptcRolloff[amp]

mu = np.linspace(0, XLIM[1], 800)

plt.figure(figsize=(6, 4))
plt.scatter(ptc.rawMeans[amp], ptc.rawVars[amp], s=1, c="0.6", label="raw")
plt.scatter(ptc.finalMeans[amp], ptc.finalVars[amp], s=1, c="k", label="final")
plt.plot(mu, funcAstier(pars, mu), label="prelim. (exp. only)")
if np.isfinite(tau) and np.isfinite(mu_star):
    plt.plot(mu, funcAstierWithRolloff(np.r_[pars, mu_star, tau], mu), label="prelim. + roll-off")
plt.plot(mu, _final_c00_model(ptc, amp, mu), ls="--", label="final model $C_{00}$")
plt.axvline(ptc.ptcTurnoff[amp], linestyle="--", color="k", alpha=0.5, label="turnoff")
plt.axvline(mu_star, linestyle=":", color="k", alpha=0.5, label="roll-off")
plt.xlim(*XLIM)
plt.ylim(*YLIM)
plt.xlabel(r"$\mu$ (ADU)")
plt.ylabel(r"$C_{00}$ (ADU$^2$)")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()